# gridMET and gridMET drought indices over the Ogallala aquifer, from the Climatology Lab's own files

Builds **one CF-1.9 NetCDF file per dataset** on Google Drive, for the whole record the provider publishes:

| Output file | Source files | Content |
|---|---|---|
| `GRIDMET_Ogallala_lab.nc` | `<variable>_<year>.nc`, 1979 → now | every daily variable (18: the 16 familiar ones plus `fm1`, `fm10` from 2020) |
| `GRIDMET_DROUGHT_Ogallala_lab.nc` | `<index>.nc` (one file per index, 1980 → now) | 26 drought indices, each with the lab's `category` layer |

Source: `https://www.northwestknowledge.net/metdata/data/` (the "direct download of NetCDF files" option of the
Climatology Lab). It is a plain file server that reports exact sizes, supports resuming and gives every file a
version tag, so each download can be verified. Nothing is taken from Earth Engine.

**How to run**

1. *Run all* with `TEST_MODE = True` (Cell 2): two years of gridMET and one drought index into `.../_test/`,
   with every check switched on and a projection of the full run.
2. Set `TEST_MODE = False` and *Run all*. If the runtime dies, run all again: both files continue from
   what is already on Drive.

**How it works** (download to scratch → verify → extract → append → delete → repeat)

* gridMET goes year by year, the drought indices file by file. Up to 10 files download at a time.
* Each file is checked against the server's size and version tag, then decoded completely by one of 6 dask
  workers, which also cuts out the aquifer window. A file that fails is downloaded again; a file that
  cannot be obtained stops the run instead of leaving a hole.
* The window is cut by index from the lab's own 1/24° grid and the lab's packed 16-bit integers are copied
  **unchanged**, with the lab's `scale_factor`, `add_offset` and `_FillValue`. (The lab's type is unsigned; the
  output uses signed 16-bit, which holds the same integers with the same bits and is what CF allows.) Readers such as xarray
  decode to physical units automatically. The whole bounding box of the aquifer is kept; `roi_mask` and
  `roi_fraction` describe the polygon.
* Every block is read back from the output file and compared before the raw files are deleted.
* The file on Drive is refreshed every hour and at the end. At least 20 GB of scratch stays free.

**What is added to the lab's data** (nothing is changed)

* `status` per time step: permanent / provisional / early, from the lab's own file attributes. A year is
  fetched again whenever the lab has changed its files, which is how provisional and early days get replaced.
* `<variable>_qc` per time step: flags records that are odd at the source (a constant field, an exact copy
  of the previous record, or no data). The lab's files do contain such records, for example wind direction
  on several 31 Decembers; they are stored as published and only flagged.
* CF metadata. The lab's original attribute texts are kept under `source_*` attributes.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q netCDF4 geopandas shapely pyproj rasterio requests "dask[distributed]" 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE    = True            # True -> short rehearsal into .../_test/ ; False -> the whole record
TEST_YEARS   = (2019, 2020)    # gridMET years exported in the rehearsal
TEST_DROUGHT = ["spi90d"]      # drought index files exported in the rehearsal (each is about 6 GB to download)

BASE_URL    = "https://www.northwestknowledge.net/metdata/data/"   # the lab's direct-download folder
RUN_GRIDMET = True             # daily meteorology  -> GRIDMET_Ogallala_lab.nc
RUN_DROUGHT = True             # drought indices    -> GRIDMET_DROUGHT_Ogallala_lab.nc
ONLY_VARS   = None             # None -> every variable the lab publishes; or e.g. ["pr", "tmmx"] / ["spi90d", "pdsi"]
YEAR_START  = None             # gridMET: None -> first year on the server; or e.g. 2000
YEAR_END    = None             # gridMET: None -> latest year on the server

N_DOWNLOAD_THREADS = 10        # files downloaded at a time
DASK_WORKERS       = 6         # worker processes that verify the files and cut out the aquifer window
WORKER_MEMORY_GB   = 3.0       # hard memory limit per dask worker
DROUGHT_GROUP      = 6         # drought files (about 6 GB each) downloaded and processed per cycle
COMPRESS_LEVEL     = 1         # zlib level of the CF .nc (lossless)
TIME_CHUNK         = 61        # time steps per HDF5 chunk of the output (the lab's files use 61 as well)

REPROCESS_CHANGED = False      # True -> also rebuild from a year stored as permanent if the lab has re-issued its files
REBUILD           = False      # True -> ignore the files on Drive and start both outputs again

MIN_FREE_GB       = 20.0       # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600   # give up (cleanly, resumable) if space never comes back
BACKUP_INTERVAL_S = 3600       # copy the file in progress to Drive this often
MAX_RETRIES       = 4          # extra download attempts per file
CROSS_CHECKS      = 6          # days read again through the lab's OPeNDAP server and compared with the output
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, CF metadata ═══════════════
import re, gc, sys, json, glob, time, math, shutil, hashlib, logging, threading, zlib, random
from datetime import datetime, timezone, timedelta, date
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import requests
import rasterio.features
from rasterio.transform import Affine
import dask
from dask.distributed import Client, LocalCluster

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a
# no-op there and INFO progress lines would never be shown.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("gridmet")
for _n in ("distributed", "urllib3.connectionpool", "rasterio", "pyogrio", "fiona"):
    logging.getLogger(_n).setLevel(logging.WARNING)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL  = "Ogallala"
DRIVE_ROOT = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT  = f"{DRIVE_ROOT}/GRIDMET_LAB" + ("/_test" if TEST_MODE else "")
SCRATCH    = "/content/gridmet_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR    = f"{SCRATCH}/raw"            # downloaded source files (deleted after use)
BLK_DIR    = f"{SCRATCH}/blocks"         # extracted windows on their way into the output
NC_DIR     = f"{SCRATCH}/nc"             # the output files being built
DASK_TMP   = f"{SCRATCH}/dask"
DISK_PATH  = "/content"                  # same filesystem as scratch AND Drive's local upload cache
for _d in (DRIVE_OUT, RAW_DIR, BLK_DIR, NC_DIR, DASK_TMP):
    os.makedirs(_d, exist_ok=True)

GRIDMET_NAME = f"GRIDMET_{ROI_LABEL}_lab.nc"
DROUGHT_NAME = f"GRIDMET_DROUGHT_{ROI_LABEL}_lab.nc"
OPENDAP_URL  = "http://thredds.northwestknowledge.net:8080/thredds/dodsC/MET/"   # used for cross-checks only

# ---------- CF constants ----------
TIME_UNITS    = "days since 1900-01-01 00:00:00"    # the lab's own time axis, copied unchanged
TIME_CALENDAR = "standard"                          # the lab writes "gregorian", the older name of the same calendar
EPOCH         = date(1900, 1, 1)
STATUS_PERMANENT, STATUS_PROVISIONAL, STATUS_EARLY, STATUS_NOT_STATED = 0, 1, 2, 3
QC_CONSTANT, QC_REPEAT, QC_NODATA = 1, 2, 4
QC_MEANINGS   = "constant_field identical_to_previous_record no_data"
LAB_CITATION  = ("Abatzoglou, J.T., 2013, Development of gridded surface meteorological data for ecological "
                 "applications and modeling, International Journal of Climatology, DOI: 10.1002/joc.3413")
LAB_PAGE      = "https://www.climatologylab.org/gridmet.html"


def _m(long_name, units, standard_name=None, cell_methods=None):
    return dict(long_name=long_name, units=units, standard_name=standard_name, cell_methods=cell_methods)


# CF wording for each lab file prefix. Descriptive only: it never touches a stored value, and the lab's
# own attribute texts are kept next to it as source_* attributes. The lab's files reuse one internal
# name for two variables (air_temperature for tmmn and tmmx, ...), so variables are named by file prefix.
# A standard_name is given only where the CF definition and its canonical units fit the variable.
# Prefixes that are not listed here are still exported, described by the lab's own attributes.
VAR_META = {
    "pr":     _m("Daily accumulated precipitation", "mm", "lwe_thickness_of_precipitation_amount", "time: sum"),
    "tmmn":   _m("Daily minimum temperature", "K", "air_temperature", "time: minimum"),
    "tmmx":   _m("Daily maximum temperature", "K", "air_temperature", "time: maximum"),
    "rmax":   _m("Daily maximum relative humidity", "%", "relative_humidity", "time: maximum"),
    "rmin":   _m("Daily minimum relative humidity", "%", "relative_humidity", "time: minimum"),
    "sph":    _m("Daily mean specific humidity", "kg kg-1", "specific_humidity", "time: mean"),
    "srad":   _m("Daily mean downward shortwave radiation at the surface", "W m-2",
                 "surface_downwelling_shortwave_flux_in_air", "time: mean"),
    "th":     _m("Daily mean wind direction, degrees clockwise from north", "degree", "wind_from_direction",
                 "time: mean"),
    "vs":     _m("Daily mean wind speed at 10 m", "m s-1", "wind_speed", "time: mean"),
    "vpd":    _m("Daily mean vapor pressure deficit", "kPa", "water_vapor_saturation_deficit_in_air", "time: mean"),
    "pet":    _m("Daily reference evapotranspiration, short grass", "mm", None, "time: sum"),
    "etr":    _m("Daily reference evapotranspiration, alfalfa", "mm", None, "time: sum"),
    "bi":     _m("Burning index (BI-G)", "1"),
    "erc":    _m("Energy release component (ERC-G)", "1"),
    "fm1":    _m("1-hour dead fuel moisture", "%"),
    "fm10":   _m("10-hour dead fuel moisture", "%"),
    "fm100":  _m("100-hour dead fuel moisture", "%"),
    "fm1000": _m("1000-hour dead fuel moisture", "%"),
    "pdsi":   _m("Palmer Drought Severity Index", "1"),
    "z":      _m("Palmer Z Index", "1"),
}
_WINDOWS = {"14d": "14 days", "30d": "30 days", "90d": "90 days", "180d": "180 days", "270d": "270 days",
            "1y": "1 year", "2y": "2 years", "5y": "5 years"}
for _p, _t in (("spi", "Standardized Precipitation Index, precipitation aggregated over the last "),
               ("spei", "Standardized Precipitation Evapotranspiration Index, climatic water balance aggregated "
                        "over the last "),
               ("eddi", "Evaporative Demand Drought Index, reference evapotranspiration aggregated over the last ")):
    for _w, _wt in _WINDOWS.items():
        VAR_META[_p + _w] = _m(_t + _wt, "1")

_UNITS_FALLBACK = {"": "1", "unitless": "1", "percent": "%", "kg/kg": "kg kg-1", "m/s": "m s-1"}


def cf_units(prefix, lab_units):
    """CF units for a variable: the table above, else the lab's units tidied where they are not udunits."""
    if prefix in VAR_META:
        return VAR_META[prefix]["units"]
    u = (lab_units or "").strip()
    return _UNITS_FALLBACK.get(u.lower(), u)


class PipelineStop(RuntimeError):
    """Clean stop: everything written so far is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC everywhere


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def day_to_date(d):
    return EPOCH + timedelta(days=int(d))


def year_of(d):
    return day_to_date(d).year

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: what the lab publishes, verified downloads, the aquifer window ═══════════════
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "gridmet-ogallala-colab/1.0 (research use)"
_adapter = requests.adapters.HTTPAdapter(pool_connections=N_DOWNLOAD_THREADS + 4, pool_maxsize=N_DOWNLOAD_THREADS + 4)
SESSION.mount("https://", _adapter)
SESSION.mount("http://", _adapter)


def list_source():
    """Read the lab's folder listing. Returns (yearly, single):
    yearly = {prefix: {year: file name}} for the daily variables (<prefix>_<year>.nc),
    single = {prefix: file name} for files that hold a whole record (<prefix>.nc, the drought indices)."""
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            r = SESSION.get(BASE_URL, timeout=(30, 180))
            r.raise_for_status()
            names = sorted(set(re.findall(r'href="([^"/?]+\.nc)"', r.text)))
            if names:
                break
            last = "no .nc links in the listing"
        except Exception as e:
            last = e
        time.sleep(10 * attempt)
    else:
        raise PipelineStop(f"Cannot read the lab's file listing at {BASE_URL}: {last}")
    yearly, single = {}, {}
    for n in names:
        m = re.fullmatch(r"(.+)_(\d{4})\.nc", n)
        if m:
            yearly.setdefault(m.group(1), {})[int(m.group(2))] = n
        else:
            single[n[:-3]] = n
    # a daily variable has a file for (nearly) every year; one-off products (e.g. chill_hours_2025) do not
    yearly = {p: ys for p, ys in yearly.items() if len(ys) >= 2}
    return yearly, single


def head_info(name):
    """Size and version tag of a file on the server (no download)."""
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            r = SESSION.head(BASE_URL + name, allow_redirects=True, timeout=(30, 60))
            r.raise_for_status()
            size = int(r.headers["Content-Length"])
            return dict(name=name, size=size, etag=r.headers.get("ETag", ""),
                        last_modified=r.headers.get("Last-Modified", ""))
        except Exception as e:
            last = e
            time.sleep(5 * attempt)
    raise PipelineStop(f"Cannot query {name} on the lab's server: {last}. Nothing is lost - re-run to resume.")


def _same_version(a, b):
    return a["size"] == b["size"] and a["etag"] == b["etag"] and a["last_modified"] == b["last_modified"]


def download_one(name):
    """Download one file to RAW_DIR, resuming a partial file, and prove it is the complete, current one:
    size equals the server's Content-Length, the version tag is the same before and after (the lab rewrites
    current files in place), and the file starts like a NetCDF file. Returns the server info."""
    dest = os.path.join(RAW_DIR, name)
    part = dest + ".part"
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            info = head_info(name)
            if os.path.exists(dest):
                if os.path.getsize(dest) == info["size"]:
                    return info
                os.remove(dest)
            pos = os.path.getsize(part) if os.path.exists(part) else 0
            if pos > info["size"]:
                os.remove(part)
                pos = 0
            if pos < info["size"]:
                headers = {}
                if pos:
                    headers["Range"] = f"bytes={pos}-"
                    if info["etag"]:
                        headers["If-Range"] = info["etag"]     # a changed file is sent whole, not spliced
                with SESSION.get(BASE_URL + name, headers=headers, stream=True, timeout=(30, 180)) as r:
                    if r.status_code == 200:
                        pos = 0
                    elif r.status_code != 206:
                        r.raise_for_status()
                        raise IOError(f"unexpected HTTP {r.status_code}")
                    with open(part, "ab" if pos else "wb") as f:
                        for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                            if chunk:
                                f.write(chunk)
                        f.flush()
                        os.fsync(f.fileno())
            got = os.path.getsize(part)
            if got != info["size"]:
                raise IOError(f"incomplete: {got} of {info['size']} bytes")
            with open(part, "rb") as f:
                magic = f.read(4)
            if magic not in (b"\x89HDF", b"CDF\x01", b"CDF\x02", b"CDF\x05"):
                os.remove(part)
                raise IOError(f"not a NetCDF file (starts with {magic!r})")
            after = head_info(name)
            if not _same_version(info, after):
                os.remove(part)
                raise IOError("the lab replaced the file during the download")
            os.replace(part, dest)
            return info
        except PipelineStop:
            raise
        except Exception as e:
            last = e
            log.warning("download %s attempt %d/%d: %s", name, attempt, MAX_RETRIES + 1, e)
            time.sleep(min(60, 5 * attempt))
    raise PipelineStop(f"{name} could not be downloaded after {MAX_RETRIES + 1} attempts ({last}). "
                       f"Nothing is lost - re-run to resume.")


def download_group(names):
    """Download several files, N_DOWNLOAD_THREADS at a time, never eating into the scratch floor."""
    with timed("download"):
        with ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
            heads = list(ex.map(head_info, names))
        need = sum(h["size"] for h in heads if not os.path.exists(os.path.join(RAW_DIR, h["name"])))
        ensure_disk_free("download", extra_gb=need / 1024 ** 3)
        t0 = time.time()
        with ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
            infos = list(ex.map(download_one, names))
        dt = max(1e-6, time.time() - t0)
        if need:
            log.info("Downloaded %d file(s), %s in %.0f s (%.1f MB/s)", len(names), human(need), dt, need / 1e6 / dt)
        TOTALS["bytes_downloaded"] = TOTALS.get("bytes_downloaded", 0) + need
    return {i["name"]: i for i in infos}


def remove_raw(names):
    for n in names:
        for p in (os.path.join(RAW_DIR, n), os.path.join(RAW_DIR, n + ".part")):
            try:
                os.remove(p)
            except OSError:
                pass


# ---------- the aquifer polygon and its window on the lab's grid ----------
def load_roi():
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs is None:
        raise DataIntegrityError("The aquifer shapefile has no CRS.")
    gdf = gdf.to_crs(epsg=4326)
    return gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union


def build_window(path, roi):
    """Read the lab's grid from one of its files and cut the smallest index window that covers the polygon.
    The window is rows/columns of the lab's own grid, so every output cell IS a lab cell."""
    with nc4.Dataset(path) as ds:
        lat = np.asarray(ds["lat"][:], "f8")
        lon = np.asarray(ds["lon"][:], "f8")
    dlat, dlon = np.diff(lat), np.diff(lon)
    if not (np.all(dlat < 0) and np.all(dlon > 0) and np.ptp(dlat) < 1e-9 and np.ptp(dlon) < 1e-9):
        raise DataIntegrityError("The lab's grid is not the expected regular north-up grid.")
    dy, dx = float(-dlat.mean()), float(dlon.mean())
    minx, miny, maxx, maxy = roi.bounds
    rows = np.where((lat - dy / 2 < maxy) & (lat + dy / 2 > miny))[0]      # cells whose extent meets the bbox
    cols = np.where((lon - dx / 2 < maxx) & (lon + dx / 2 > minx))[0]
    if rows.size == 0 or cols.size == 0:
        raise DataIntegrityError("The aquifer polygon does not overlap the lab's grid.")
    r0, r1, c0, c1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    H, W = r1 - r0, c1 - c0
    wlat, wlon = lat[r0:r1], lon[c0:c1]
    left, top = float(wlon[0] - dx / 2), float(wlat[0] + dy / 2)
    aff = Affine(dx, 0, left, 0, -dy, top)
    touch = rasterio.features.geometry_mask([roi], out_shape=(H, W), transform=aff, invert=True, all_touched=True)
    S = 20                                             # share of each cell inside the polygon, from 20x20 sub-cells
    fine = rasterio.features.geometry_mask([roi], out_shape=(H * S, W * S), invert=True,
                                           transform=Affine(dx / S, 0, left, 0, -dy / S, top))
    frac = fine.reshape(H, S, W, S).mean(axis=(1, 3)).astype("float32")
    win = dict(r0=r0, r1=r1, c0=c0, c1=c1, H=H, W=W, dx=dx, dy=dy, left=left, top=top,
               lat=wlat, lon=wlon, full_shape=(int(lat.size), int(lon.size)),
               roi_mask=touch.astype("int8"), roi_fraction=frac)
    log.info("Lab grid %d x %d, step %.6f deg | aquifer window rows %d-%d, cols %d-%d = %d x %d cells "
             "(%.1f %% touch the polygon)", lat.size, lon.size, dx, r0, r1 - 1, c0, c1 - 1, H, W, 100 * touch.mean())
    return win


def worker_spec(win):
    """The part of the window a dask worker needs (plain Python types only)."""
    return dict(r0=win["r0"], r1=win["r1"], c0=win["c0"], c1=win["c1"], full_shape=tuple(win["full_shape"]),
                lat=[float(v) for v in win["lat"]], lon=[float(v) for v in win["lon"]], out_dir=BLK_DIR)

In [ ]:
# ═══════════════ CELL 6: verify one source file and cut out the aquifer window (runs in dask workers) ═══════════════
def extract_file(path, prefix, spec):
    """Decode a whole lab file (which proves every compressed chunk is readable), check that it is on the
    expected grid, and copy the aquifer window of every (day, lat, lon) variable to a .npy file in the lab's
    own packed integers. No value is converted: auto mask/scale is off and the window is a plain index slice.
    The lab stores unsigned 16-bit integers that never exceed 32767 (its fill value); they are written as
    signed 16-bit, which is the same integers and the same bits and is what CF allows for packed data.
    A file with a larger value is rejected rather than reinterpreted.

    Returns a plain dict: ok/error, the day axis, the lab's global attributes and, per variable, the block
    path, its attributes and one row of statistics per day (valid cells, constant?, CRC32)."""
    import os, zlib
    import numpy as np
    import netCDF4 as nc4

    def plain(v):
        if isinstance(v, np.ndarray):
            return v.tolist()
        if isinstance(v, np.generic):
            return v.item()
        return v

    res = dict(file=os.path.basename(path), prefix=prefix, ok=False, blocks=[])
    try:
        ds = nc4.Dataset(path)
        try:
            ds.set_auto_maskandscale(False)
            for d in ("day", "lat", "lon"):
                if d not in ds.dimensions:
                    raise ValueError(f"dimension '{d}' is missing")
            lat = np.asarray(ds["lat"][:], "f8")
            lon = np.asarray(ds["lon"][:], "f8")
            r0, r1, c0, c1 = spec["r0"], spec["r1"], spec["c0"], spec["c1"]
            if (lat.size, lon.size) != tuple(spec["full_shape"]):
                raise ValueError(f"grid is {lat.size} x {lon.size}, expected {spec['full_shape']}")
            if not (np.array_equal(lat[r0:r1], np.asarray(spec["lat"])) and
                    np.array_equal(lon[c0:c1], np.asarray(spec["lon"]))):
                raise ValueError("lat/lon of the aquifer window differ from the reference grid")
            days = np.asarray(ds["day"][:], "f8")
            if days.size == 0 or not np.all(np.isfinite(days)) or np.any(np.diff(days) <= 0) \
                    or np.any(days != np.round(days)):
                raise ValueError("day axis is empty, non-finite, fractional or not strictly increasing")
            tunits = str(getattr(ds["day"], "units", ""))
            if tunits.replace("T", " ").strip() not in ("days since 1900-01-01 00:00:00", "days since 1900-01-01"):
                raise ValueError(f"unexpected time units {tunits!r}")
            res["days"] = days.tolist()
            res["global"] = {k: plain(ds.getncattr(k)) for k in ds.ncattrs()}
            names = [k for k, v in ds.variables.items() if v.dimensions == ("day", "lat", "lon")]
            if not names:
                raise ValueError("no (day, lat, lon) variable")
            n, H, W = days.size, r1 - r0, c1 - c0
            plane = lat.size * lon.size
            for name in names:
                v = ds[name]
                if v.dtype.kind not in "iu":
                    raise ValueError(f"{name}: expected packed integers, found {v.dtype}")
                fill = v.getncattr("_FillValue") if "_FillValue" in v.ncattrs() else None
                sdtype = np.dtype(f"i{v.dtype.itemsize}")          # signed type of the same width
                smax = np.iinfo(sdtype).max
                ch = v.chunking()
                step = ch[0] if isinstance(ch, (list, tuple)) else 32
                step = max(1, min(step, (640 * 1024 * 1024) // (plane * v.dtype.itemsize)))
                out = os.path.join(spec["out_dir"], f"{res['file'][:-3]}__{name}.npy")
                mm = np.lib.format.open_memmap(out, mode="w+", dtype=sdtype, shape=(n, H, W))
                nvalid = np.zeros(n, "i8")
                const = np.zeros(n, bool)
                crc = np.zeros(n, "u8")
                for i in range(0, n, step):
                    a = v[i:i + step]                      # whole grid: every chunk of the file is decoded
                    if a.shape[1:] != (lat.size, lon.size):
                        raise ValueError(f"{name}: slab shape {a.shape}")
                    if v.dtype.kind == "u" and a.size and int(a.max()) > smax:
                        raise ValueError(f"{name}: value {int(a.max())} does not fit signed {sdtype}")
                    w = np.ascontiguousarray(a[:, r0:r1, c0:c1]).astype(sdtype, copy=False)
                    mm[i:i + w.shape[0]] = w
                    for k in range(w.shape[0]):
                        vals = w[k] if fill is None else w[k][w[k] != fill]
                        nvalid[i + k] = vals.size
                        const[i + k] = vals.size > 1 and vals.min() == vals.max()
                        crc[i + k] = zlib.crc32(w[k].tobytes())
                    del a, w
                mm.flush()
                del mm
                res["blocks"].append(dict(name=name, path=out, dtype=str(sdtype), source_dtype=str(v.dtype),
                                          shape=[n, H, W],
                                          attrs={k: plain(v.getncattr(k)) for k in v.ncattrs()},
                                          nvalid=nvalid.tolist(), const=const.tolist(), crc=crc.tolist()))
            res["ok"] = True
        finally:
            ds.close()
    except Exception as e:
        res["error"] = f"{type(e).__name__}: {e}"
        for b in res.get("blocks", []):
            try:
                os.remove(b["path"])
            except OSError:
                pass
        res["blocks"] = []
    return res


def slice_status(glob_attrs, n, default=STATUS_PERMANENT):
    """Per-day status of one lab file from its own attributes (1-based slice numbers):
    days after last_permanent_slice are provisional, days after last_provisional_slice are early.
    A file without these attributes gets `default`: permanent for a closed gridMET year (the lab removes
    the attributes when a year is final), not_stated for the drought files, which never carry them."""
    st = np.full(n, default, "i1")

    def num(key):
        try:
            return int(str(glob_attrs[key]).strip())
        except (KeyError, ValueError):
            return None

    perm, prov = num("last_permanent_slice"), num("last_provisional_slice")
    if perm is not None:
        st[:] = STATUS_PERMANENT
        st[max(0, perm):] = STATUS_PROVISIONAL
    if prov is not None:
        st[max(0, prov):] = STATUS_EARLY
    return st


def start_cluster():
    cluster = LocalCluster(n_workers=DASK_WORKERS, threads_per_worker=1, processes=True,
                           memory_limit=f"{WORKER_MEMORY_GB}GB", local_directory=DASK_TMP, dashboard_address=None)
    client = Client(cluster)
    log.info("dask: %d worker processes, %.1f GB each", DASK_WORKERS, WORKER_MEMORY_GB)
    return cluster, client


def extract_group(client, jobs, win):
    """jobs = [(file name, prefix)]. Verify + extract in parallel; a file that fails is downloaded again
    once and retried. Returns {file name: result}. Raises PipelineStop if a file stays unreadable."""
    spec = worker_spec(win)
    with timed("verify + extract"):
        futs = {name: client.submit(extract_file, os.path.join(RAW_DIR, name), prefix, spec, pure=False)
                for name, prefix in jobs}
        out = {name: f.result() for name, f in futs.items()}
    bad = [(name, prefix) for name, prefix in jobs if not out[name]["ok"]]
    for name, prefix in bad:
        log.warning("CORRUPT or unexpected: %s (%s) - downloading it again", name, out[name].get("error"))
        log_events("corrupted_files.jsonl", [dict(file=name, error=out[name].get("error"))])
        remove_raw([name])
        download_group([name])
        with timed("verify + extract"):
            out[name] = client.submit(extract_file, os.path.join(RAW_DIR, name), prefix, spec, pure=False).result()
        if not out[name]["ok"]:
            raise PipelineStop(f"{name} is unreadable after a fresh download ({out[name].get('error')}). "
                               f"The output is unchanged for this step - re-run later to resume.")
    return out

In [ ]:
# ═══════════════ CELL 7: the CF output file (create, add variables, write blocks, read back) ═══════════════
KIND_TEXT = {
    "gridmet": dict(
        title="gridMET daily surface meteorology over the Ogallala (High Plains) aquifer",
        summary="All daily variables of the gridMET product of the Climatology Lab, cut by index from the lab's "
                "yearly NetCDF files to the bounding box of the High Plains aquifer on the lab's own 1/24 degree "
                "grid. Values are the lab's packed 16-bit integers, copied unchanged with the lab's "
                "scale_factor, add_offset and _FillValue.",
        files="<variable>_<year>.nc"),
    "drought": dict(
        title="gridMET drought indices over the Ogallala (High Plains) aquifer",
        summary="All drought indices of the Climatology Lab's gridMET drought product (SPI, SPEI, EDDI at eight "
                "time scales, PDSI and Palmer Z, each with the lab's drought category layer), cut by index from "
                "the lab's NetCDF files to the bounding box of the High Plains aquifer on the lab's own 1/24 "
                "degree grid. Values are the lab's packed 16-bit integers, copied unchanged with the lab's "
                "scale_factor, add_offset and _FillValue.",
        files="<index>.nc"),
}


def raw(ds):
    """Make every variable of an open dataset return and accept stored values exactly (no mask, no scale)."""
    ds.set_auto_maskandscale(False)
    return ds


def create_output(path, kind, win):
    H, W, dx, dy = win["H"], win["W"], win["dx"], win["dy"]
    txt = KIND_TEXT[kind]
    with nc4.Dataset(path, "w", format="NETCDF4") as ds:
        ds.createDimension("time", None)
        ds.createDimension("lat", H)
        ds.createDimension("lon", W)
        ds.createDimension("nv", 2)

        t = ds.createVariable("time", "f8", ("time",), chunksizes=(1024,))
        t.setncatts(dict(standard_name="time", long_name="time", units=TIME_UNITS, calendar=TIME_CALENDAR, axis="T",
                         comment="Copied from the lab's 'day' axis. Per the lab: days correspond approximately to "
                                 "calendar days ending at midnight Mountain Standard Time (7 UTC the next "
                                 "calendar day)."))
        la = ds.createVariable("lat", "f8", ("lat",))
        la.setncatts(dict(standard_name="latitude", long_name="latitude of cell centre", units="degrees_north",
                          axis="Y", bounds="lat_bnds"))
        lo = ds.createVariable("lon", "f8", ("lon",))
        lo.setncatts(dict(standard_name="longitude", long_name="longitude of cell centre", units="degrees_east",
                          axis="X", bounds="lon_bnds"))
        la[:] = win["lat"]
        lo[:] = win["lon"]
        lb = ds.createVariable("lat_bnds", "f8", ("lat", "nv"))
        lb[:] = np.stack([win["lat"] + dy / 2, win["lat"] - dy / 2], axis=1)
        ob = ds.createVariable("lon_bnds", "f8", ("lon", "nv"))
        ob[:] = np.stack([win["lon"] - dx / 2, win["lon"] + dx / 2], axis=1)

        crs = ds.createVariable("crs", "i4", ())
        crs.setncatts(dict(grid_mapping_name="latitude_longitude", long_name="WGS 84",
                           semi_major_axis=6378137.0, inverse_flattening=298.257223563,
                           longitude_of_prime_meridian=0.0, crs_wkt=pyproj.CRS.from_epsg(4326).to_wkt(),
                           GeoTransform=f"{win['left']!r} {dx!r} 0 {win['top']!r} 0 {-dy!r}",
                           comment="The lab states: the projection information for its files is GCS WGS 1984."))
        crs.assignValue(0)

        m = ds.createVariable("roi_mask", "i1", ("lat", "lon"), zlib=True, complevel=COMPRESS_LEVEL)
        m.setncatts(dict(long_name="cells touched by the High Plains aquifer polygon (hp_bound2010)",
                         flag_values=np.array([0, 1], "i1"), flag_meanings="not_touched touched",
                         grid_mapping="crs", comment="Descriptive only: data are stored for every cell."))
        m[:] = win["roi_mask"]
        f = ds.createVariable("roi_fraction", "f4", ("lat", "lon"), zlib=True, complevel=COMPRESS_LEVEL)
        f.setncatts(dict(long_name="share of the cell area inside the High Plains aquifer polygon", units="1",
                         grid_mapping="crs", comment="Estimated from 20 x 20 sub-cells per cell."))
        f[:] = win["roi_fraction"]

        s = ds.createVariable("status", "i1", ("time",), chunksizes=(1024,))
        s.setncatts(dict(long_name="provider status of the record", standard_name="status_flag",
                         flag_values=np.array([0, 1, 2, 3], "i1"),
                         flag_meanings="permanent provisional early not_stated",
                         comment="From the lab's file attributes last_permanent_slice / last_provisional_slice. "
                                 "Stored as the lab states it. Re-running the notebook fetches again every year whose "
                                 "files the lab has changed. Where variables differ, the least final status is kept. "
                                 "not_stated: the lab's file carries no status information (drought files)."))

        now = iso(utcnow())
        ds.setncatts(dict(
            Conventions="CF-1.9", title=txt["title"], summary=txt["summary"],
            institution="Climatology Lab, University of California Merced (data producer); extracted by the "
                        "MSU groundwater-buffering project",
            source=f"gridMET NetCDF files {txt['files']} downloaded from {BASE_URL}",
            references=f"{LAB_CITATION}; {LAB_PAGE}",
            license="Public domain per the provider; cite the reference above and note the date of acquisition.",
            history=f"{now} created by the Colab notebook GridMET_Lab_Ogallala.ipynb",
            comment="Each output cell is one cell of the lab's grid (index window, no resampling). "
                    "<variable>_qc flags records that are unusual at the source; no value is altered.",
            product_kind=kind, roi_name=f"{ROI_LABEL} (hp_bound2010), bounding box",
            window_rows=f"{win['r0']}:{win['r1']}", window_cols=f"{win['c0']}:{win['c1']}",
            source_grid_shape=f"{win['full_shape'][0]} x {win['full_shape'][1]}",
            geospatial_lat_min=float(win["top"] - H * dy), geospatial_lat_max=float(win["top"]),
            geospatial_lon_min=float(win["left"]), geospatial_lon_max=float(win["left"] + W * dx),
            geospatial_lat_resolution=dy, geospatial_lon_resolution=dx,
            geospatial_lat_units="degrees_north", geospatial_lon_units="degrees_east",
            date_created=now, date_modified=now, processing_status="in_progress",
            committed_records=np.int32(0), requested_variables=""))


def check_window(ds, win):
    """The output file must be on exactly the window computed from today's lab grid."""
    if ds.dimensions["lat"].size != win["H"] or ds.dimensions["lon"].size != win["W"] \
            or not np.array_equal(ds["lat"][:], win["lat"]) or not np.array_equal(ds["lon"][:], win["lon"]):
        raise PipelineStop("The file on Drive is on a different grid window than the lab's current grid / the "
                           "current shapefile. Set REBUILD = True to start it again.")


def output_names(prefix, block_names):
    """Name of each (day, lat, lon) variable of a lab file in the output, and its role."""
    data = [b for b in block_names if b != "category"]
    out = {}
    for b in block_names:
        if b == "category":
            out[b] = (f"{prefix}_category", "category")
        elif len(data) == 1:
            out[b] = (prefix, "data")
        else:
            out[b] = (f"{prefix}_{b}", "data")
    return out


_PACKING = ("scale_factor", "add_offset", "_FillValue")


def ensure_var(ds, out_name, role, prefix, block, source_file):
    """Return the output variable for a block, creating it (and its _qc companion) on first use.
    The lab's packing must be identical in every file that feeds one variable, otherwise copying the
    integers unchanged would silently change their meaning: that case stops the run."""
    a = block["attrs"]
    dtype = np.dtype(block["dtype"])
    if out_name in ds.variables:
        v = ds[out_name]
        if v.dtype != dtype:
            raise DataIntegrityError(f"{source_file}: {out_name} is {dtype} but the output holds {v.dtype}.")
        for k in _PACKING:
            have = v.getncattr(k) if k in v.ncattrs() else None
            want = a.get(k)
            if (have is None) != (want is None) or (have is not None and float(have) != float(want)):
                raise DataIntegrityError(f"{source_file}: {k} of {out_name} is {want!r} but earlier files had "
                                         f"{have!r}. One packed variable cannot hold both.")
        return v
    H, W = ds.dimensions["lat"].size, ds.dimensions["lon"].size
    fill = dtype.type(a["_FillValue"]) if "_FillValue" in a else None
    v = ds.createVariable(out_name, dtype, ("time", "lat", "lon"), zlib=COMPRESS_LEVEL > 0,
                          complevel=COMPRESS_LEVEL, shuffle=True, chunksizes=(TIME_CHUNK, H, W), fill_value=fill)
    v.set_auto_maskandscale(False)
    meta = VAR_META.get(prefix, {})
    lab_desc = str(a.get("description", "")).strip()
    att = {}
    if role == "category":
        att["long_name"] = f"Drought category of {prefix}"
        att["units"] = "1"
    else:
        att["long_name"] = meta.get("long_name") or lab_desc or prefix
        att["units"] = cf_units(prefix, a.get("units"))
        if meta.get("standard_name"):
            att["standard_name"] = meta["standard_name"]
        if meta.get("cell_methods"):
            att["cell_methods"] = meta["cell_methods"]
        for k in ("scale_factor", "add_offset"):
            if k in a:
                att[k] = np.float64(a[k])
        att["ancillary_variables"] = f"{out_name}_qc status"
    att["grid_mapping"] = "crs"
    if lab_desc:
        att["description"] = lab_desc
    att["source_variable"] = block["name"]
    att["source_dtype"] = block.get("source_dtype", block["dtype"])
    att["source_file_pattern"] = re.sub(r"_\d{4}\.nc$", "_<year>.nc", source_file)
    for k in ("units", "long_name", "standard_name"):
        if k in a:
            att[f"source_{k}"] = str(a[k])
    v.setncatts(att)
    if role == "data":
        q = ds.createVariable(f"{out_name}_qc", "u1", ("time",), chunksizes=(1024,))
        q.set_auto_maskandscale(False)
        q.setncatts(dict(long_name=f"source quality flags of {out_name}", standard_name="quality_flag",
                         flag_masks=np.array([QC_CONSTANT, QC_REPEAT, QC_NODATA], "u1"), flag_meanings=QC_MEANINGS,
                         comment="0 = nothing unusual. constant_field: all valid cells of the window hold one "
                                 "value (natural for precipitation on a dry day). identical_to_previous_record: the window equals the previous record "
                                 "exactly. no_data: the record is entirely _FillValue or absent at the source. "
                                 "Flags describe the lab's data; no value is altered."))
        n = ds.dimensions["time"].size
        if n:
            q[0:n] = np.full(n, QC_NODATA, "u1")
    return v


def _runs(rows):
    """Split sorted row indices into contiguous runs: [(first position, first row, length)]."""
    rows = np.asarray(rows, "i8")
    cuts = np.where(np.diff(rows) != 1)[0] + 1
    out, start = [], 0
    for c in list(cuts) + [rows.size]:
        out.append((start, int(rows[start]), int(c - start)))
        start = int(c)
    return out


def write_block(v, rows, path):
    """Copy a block (.npy, one slab per source day) into output rows `rows`, a few chunks at a time."""
    mm = np.load(path, mmap_mode="r")
    step = max(1, TIME_CHUNK * 4)
    for pos, row, length in _runs(rows):
        for i in range(0, length, step):
            k = min(step, length - i)
            v[row + i:row + i + k] = mm[pos + i:pos + i + k]
    del mm


def verify_block(v, rows, path):
    """Read the rows back from the output and compare them with the block, bit for bit."""
    mm = np.load(path, mmap_mode="r")
    step = max(1, TIME_CHUNK * 4)
    for pos, row, length in _runs(rows):
        for i in range(0, length, step):
            k = min(step, length - i)
            if not np.array_equal(v[row + i:row + i + k], mm[pos + i:pos + i + k]):
                del mm
                raise DataIntegrityError(f"Read-back of {v.name} rows {row + i}..{row + i + k - 1} differs from the "
                                         f"source block. The raw files were kept; re-run to repeat this step.")
    del mm


def qc_flags(block, prev_crc=None):
    """QC bits for the days of one block. prev_crc = CRC of the record just before the block (or None)."""
    nvalid = np.asarray(block["nvalid"])
    crc = np.asarray(block["crc"], "u8")
    q = np.zeros(nvalid.size, "u1")
    q[np.asarray(block["const"], bool)] |= QC_CONSTANT
    q[nvalid == 0] |= QC_NODATA
    rep = np.zeros(nvalid.size, bool)
    rep[1:] = crc[1:] == crc[:-1]
    if prev_crc is not None and nvalid.size:
        rep[0] = crc[0] == prev_crc
    q[rep & (nvalid > 0)] |= QC_REPEAT
    return q


def remove_blocks(results):
    for r in results.values():
        for b in r.get("blocks", []):
            try:
                os.remove(b["path"])
            except OSError:
                pass


def backup(local, drive_path, extra=()):
    """Verified copy of the (closed) output file and its small companions to Drive."""
    robust_drive_copy(local, drive_path)
    for src in extra:
        if os.path.exists(src):
            shutil.copyfile(src, os.path.join(DRIVE_OUT, os.path.basename(src)))


def stage_output(name):
    """Decide which copy of an output to continue from: scratch if it is at least as far along as Drive,
    otherwise Drive. Returns the local path (the file may not exist yet)."""
    local, on_drive = os.path.join(NC_DIR, name), os.path.join(DRIVE_OUT, name)
    if REBUILD:
        if os.path.exists(local):
            os.remove(local)
        log.warning("REBUILD = True: %s starts again from the first record.", name)
        return local

    def progress(p):
        try:
            with nc4.Dataset(p) as ds:
                return int(getattr(ds, "committed_records", 0)), str(getattr(ds, "date_modified", ""))
        except Exception:
            return None

    pl = progress(local) if os.path.exists(local) else None
    pd_ = progress(on_drive) if os.path.exists(on_drive) else None
    if os.path.exists(on_drive) and pd_ is None:
        raise PipelineStop(f"{on_drive} exists but cannot be opened. Move it away (or set REBUILD = True) and re-run.")
    if pd_ is not None and (pl is None or pl < pd_):
        restore_from_drive(on_drive, local)
    elif pl is None and os.path.exists(local):
        os.remove(local)                                  # unreadable scratch leftover, nothing on Drive
    return local

In [ ]:
# ═══════════════ CELL 8: gridMET, year by year ═══════════════
def _days_in_year(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def _select(prefixes):
    """The lab's prefixes to export: all of them, or those named in ONLY_VARS."""
    return sorted(p for p in prefixes if ONLY_VARS is None or p in ONLY_VARS)


def _time_size(path):
    with nc4.Dataset(path) as ds:
        return ds.dimensions["time"].size


def gridmet_resume_point(local, years, variables, sidecar, heads_by_year):
    """Where to continue. A stored year is kept when the lab's files for it are still the versions it was
    built from (size + version tag, remembered in the source list next to the output). The file is rewound
    to the first year that is missing, incomplete, or whose files the lab has changed since - which is how
    provisional and early days get replaced. Returns (records to keep, first year to process or None)."""
    if not os.path.exists(local):
        return 0, years[0]
    with nc4.Dataset(local) as ds:
        raw(ds)
        if str(getattr(ds, "product_kind", "")) != "gridmet":
            raise PipelineStop(f"{local} is not a gridMET output of this notebook.")
        want = ",".join(variables)
        have = str(getattr(ds, "requested_variables", ""))
        if have and have != want:
            raise PipelineStop(f"The file on Drive was built for variables [{have}] but this run asks for "
                               f"[{want}]. Restore ONLY_VARS, or set REBUILD = True.")
        n = min(int(getattr(ds, "committed_records", 0)), ds.dimensions["time"].size)
        t = np.asarray(ds["time"][:n], "f8")
        st = np.asarray(ds["status"][:n])
    if n == 0:
        return 0, years[0]
    if np.any(np.diff(t) <= 0):
        raise PipelineStop("The time axis of the file on Drive is not increasing. Set REBUILD = True.")
    yrs = np.array([year_of(d) for d in t])
    if yrs[0] != years[0]:
        raise PipelineStop(f"The file on Drive starts in {yrs[0]} but this run starts in {years[0]}. "
                           f"Restore YEAR_START, or set REBUILD = True.")
    first_open, reissued = None, []
    for y in years:
        sel = yrs == y
        cnt = int(sel.sum())
        files = heads_by_year[y]
        rec = sidecar.get("years", {}).get(str(y))
        if rec is None or rec.get("records") != cnt or cnt == 0:
            # no record of what this year was built from: trust it only if it is complete and permanent
            if not (cnt == _days_in_year(y) and np.all(st[sel] == STATUS_PERMANENT)):
                first_open = y
                break
            continue
        if set(rec.get("files", [])) != set(files):
            first_open = y                                 # the lab added or removed a variable for this year
            break
        changed = [f for f, h in files.items() if not _same_version(sidecar["files"].get(f, {}), h)]
        if changed:
            if REPROCESS_CHANGED or np.any(st[sel] != STATUS_PERMANENT):
                first_open = y
                break
            reissued.append((y, changed[0]))
    if reissued:
        log.warning("The lab re-issued files of %d year(s) that are stored as permanent, e.g. %s. Set "
                    "REPROCESS_CHANGED = True to rebuild from %d.", len(reissued), reissued[0][1], reissued[0][0])
    if first_open is None:
        return n, None
    return int((yrs < first_open).sum()), first_open


def run_gridmet(client, roi, yearly):
    """Download one year of every variable, verify, cut the window, append, delete, repeat."""
    variables = _select(yearly)
    if not variables:
        log.info("gridMET: no variable selected.")
        return None
    all_years = sorted(set().union(*[set(yearly[p]) for p in variables]))
    lo = TEST_YEARS[0] if TEST_MODE else (YEAR_START or all_years[0])
    hi = TEST_YEARS[1] if TEST_MODE else (YEAR_END or all_years[-1])
    years = [y for y in all_years if lo <= y <= hi]
    if not years:
        raise PipelineStop(f"gridMET: no year between {lo} and {hi} on the server.")
    log.info("gridMET: %d variables %s | years %d-%d", len(variables), variables, years[0], years[-1])

    local = stage_output(GRIDMET_NAME)
    on_drive = os.path.join(DRIVE_OUT, GRIDMET_NAME)
    side_local = os.path.join(NC_DIR, GRIDMET_NAME[:-3] + "_sources.json")
    side_drive = os.path.join(DRIVE_OUT, os.path.basename(side_local))
    sidecar = dict(files={}, years={})
    if not REBUILD and os.path.exists(side_drive):
        try:
            sidecar = json.load(open(side_drive))
        except Exception:
            log.warning("The source list next to the file on Drive is unreadable; change detection starts afresh.")
    sidecar.setdefault("files", {})
    sidecar.setdefault("years", {})

    # one HEAD per file: total volume for the projection, and detection of files the lab has changed
    names = [(y, p, yearly[p][y]) for y in years for p in variables if y in yearly[p]]
    with ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
        heads = dict(zip([n for _, _, n in names], ex.map(head_info, [n for _, _, n in names])))
    heads_by_year = {y: {} for y in years}
    for y, p, n in names:
        heads_by_year[y][n] = heads[n]
    total_bytes = sum(h["size"] for h in heads.values())

    keep, first = gridmet_resume_point(local, years, variables, sidecar, heads_by_year)
    if first is None:
        log.info("gridMET: the file already holds the lab's current version of every year (%d records). "
                 "Nothing to do.", keep)
        return dict(path=on_drive, records=keep, new_records=0, total_bytes=total_bytes, todo_bytes=0)
    todo_years = [y for y in years if y >= first]
    todo_bytes = sum(heads[n]["size"] for y, _, n in names if y >= first)
    log.info("gridMET: keeping %d records, processing %d-%d (%d files, %s of %s on the server)",
             keep, todo_years[0], todo_years[-1], sum(1 for y, _, _ in names if y >= first),
             human(todo_bytes), human(total_bytes))

    win, ds = None, None
    prev_crc = {}
    n = keep
    last_backup = time.time()
    t_run = time.time()
    try:
        for y in todo_years:
            jobs = [(yearly[p][y], p) for p in variables if y in yearly[p]]
            infos = download_group([name for name, _ in jobs])
            if win is None:
                win = build_window(os.path.join(RAW_DIR, jobs[0][0]), roi)
                if not os.path.exists(local):
                    create_output(local, "gridmet", win)
                ds = raw(nc4.Dataset(local, "a"))
                check_window(ds, win)
                ds.requested_variables = ",".join(variables)
                ds.committed_records = np.int32(n)        # records after this point are being rewritten
                if n:                                     # CRC of the last kept record, for the repeat flag
                    for name in ds.variables:
                        if name + "_qc" in ds.variables:
                            prev_crc[name] = zlib.crc32(np.ascontiguousarray(ds[name][n - 1]).tobytes())
            results = extract_group(client, jobs, win)

            days = sorted(set().union(*[set(r["days"]) for r in results.values()]))
            if any(year_of(d) != y for d in days):
                raise DataIntegrityError(f"A {y} file holds days of another year.")
            if n and days[0] <= float(ds["time"][n - 1]):
                raise DataIntegrityError(f"{y} does not continue the stored time axis.")
            days_arr = np.asarray(days, "f8")
            if np.any(np.diff(days_arr) != 1):
                log.warning("gridMET %d: the lab's files skip %d day(s) inside the year.", y,
                            int((np.diff(days_arr) - 1).sum()))
            m = days_arr.size
            status = np.zeros(m, "i1")
            written, seen = [], set()
            with timed("write"):
                ds["time"][n:n + m] = days_arr
                for name, prefix in jobs:
                    r = results[name]
                    pos = np.searchsorted(days_arr, np.asarray(r["days"], "f8"))
                    st = slice_status(r["global"], len(r["days"]))
                    status[pos] = np.maximum(status[pos], st)
                    mapping = output_names(prefix, [b["name"] for b in r["blocks"]])
                    for b in r["blocks"]:
                        out_name, role = mapping[b["name"]]
                        v = ensure_var(ds, out_name, role, prefix, b, name)
                        write_block(v, n + pos, b["path"])
                        written.append((v, n + pos, b["path"]))
                        if role == "data":
                            q = np.full(m, QC_NODATA, "u1")
                            q[pos] = qc_flags(b, prev_crc.get(out_name) if pos[0] == 0 else None)
                            ds[out_name + "_qc"][n:n + m] = q
                            prev_crc[out_name] = int(b["crc"][-1]) if pos[-1] == m - 1 else None
                            seen.add(out_name)
                for name in ds.variables:                 # variables that have no file this year
                    if name.endswith("_qc") and name[:-3] not in seen:
                        ds[name][n:n + m] = np.full(m, QC_NODATA, "u1")
                        prev_crc[name[:-3]] = None
                ds["status"][n:n + m] = status
                ds.sync()
            with timed("read-back check"):
                for v, rows, path in written:
                    verify_block(v, rows, path)
            n += m
            ds.committed_records = np.int32(n)
            ds.date_modified = iso(utcnow())
            ds.time_coverage_start = day_to_date(float(ds["time"][0])).isoformat()
            ds.time_coverage_end = day_to_date(days_arr[-1]).isoformat()
            ds.processing_status = "complete" if y == todo_years[-1] else "in_progress"
            ds.sync()
            for name, _ in jobs:
                sidecar["files"][name] = dict(infos[name], days=len(results[name]["days"]), stored=iso(utcnow()),
                                              file_date=str(results[name]["global"].get("date", "")))
            sidecar["years"][str(y)] = dict(records=int(m), files=sorted(name for name, _ in jobs))
            json.dump(sidecar, open(side_local, "w"), indent=1)
            remove_raw([name for name, _ in jobs])
            remove_blocks(results)
            TOTALS["gridmet_files"] = TOTALS.get("gridmet_files", 0) + len(jobs)
            st_txt = {k: int((status == c).sum()) for k, c in (("permanent", 0), ("provisional", 1), ("early", 2))}
            log.info("gridMET %d: %d days x %d variables appended (records %d) | %s | %.1f GB free",
                     y, m, len(jobs), n, st_txt, free_gb())
            if time.time() - last_backup >= BACKUP_INTERVAL_S and y != todo_years[-1]:
                ds.close()
                ds = None
                backup(local, on_drive, [side_local])
                ds = raw(nc4.Dataset(local, "a"))
                last_backup = time.time()
    except (KeyboardInterrupt, PipelineStop, DataIntegrityError) as e:
        log.warning("gridMET stopped (%s). Saving the %d committed records to Drive.", type(e).__name__, n)
        raise
    finally:
        if ds is not None:
            ds.close()
        if os.path.exists(local) and n > keep:
            backup(local, on_drive, [side_local])
    if n and os.path.exists(local) and _time_size(local) > n:
        log.warning("The file holds %d stale record(s) after the last committed one. Set REBUILD = True to remove "
                    "them.", _time_size(local) - n)
    return dict(path=on_drive, records=n, new_records=n - keep, total_bytes=total_bytes, todo_bytes=todo_bytes,
                seconds=time.time() - t_run, years=len(todo_years), all_years=len(all_years))

In [ ]:
# ═══════════════ CELL 9: drought indices, file by file, then RUN ═══════════════
def _drought_done(ds, prefix, head):
    """True if this index is already in the output from exactly this version of the lab's file."""
    if prefix not in ds.variables:
        return False
    v = ds[prefix]
    a = {k: v.getncattr(k) for k in v.ncattrs()}
    return (int(a.get("source_complete", 0)) == 1 and str(a.get("source_etag", "")) == head["etag"]
            and int(a.get("source_size", -1)) == head["size"]
            and str(a.get("source_last_modified", "")) == head["last_modified"])


def run_drought(client, roi, single):
    """Each index is one large file holding the whole record: download a few, verify, cut the window,
    write the index and its category layer, delete, repeat."""
    prefixes = _select(single)
    if TEST_MODE:
        prefixes = [p for p in prefixes if p in TEST_DROUGHT]
    if not prefixes:
        log.info("Drought: no index selected.")
        return None
    with ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
        heads = dict(zip(prefixes, ex.map(head_info, [single[p] for p in prefixes])))
    total_bytes = sum(h["size"] for h in heads.values())

    local = stage_output(DROUGHT_NAME)
    on_drive = os.path.join(DRIVE_OUT, DROUGHT_NAME)
    todo = list(prefixes)
    if os.path.exists(local):
        with nc4.Dataset(local) as ds:
            if str(getattr(ds, "product_kind", "")) != "drought":
                raise PipelineStop(f"{local} is not a drought output of this notebook.")
            todo = [p for p in prefixes if not _drought_done(ds, p, heads[p])]
    todo_bytes = sum(heads[p]["size"] for p in todo)
    log.info("Drought: %d index files selected, %d to process (%s of %s); %d already current in the file",
             len(prefixes), len(todo), human(todo_bytes), human(total_bytes), len(prefixes) - len(todo))
    if not todo:
        return dict(path=on_drive, records=_time_size(local), new_files=0, total_bytes=total_bytes, todo_bytes=0)

    win, ds = None, None
    status_run = {}                       # day -> least final status seen in this run
    done = 0
    last_backup = time.time()
    t_run = time.time()
    try:
        i = 0
        while i < len(todo):
            # as many files per cycle as the scratch floor allows (each is several GB)
            group, need = [], 0.0
            budget = free_gb() - MIN_FREE_GB - 2.0
            for p in todo[i:i + DROUGHT_GROUP]:
                gb = heads[p]["size"] / 1024 ** 3
                if group and need + gb > budget:
                    break
                group.append(p)
                need += gb
            i += len(group)
            jobs = [(single[p], p) for p in group]
            infos = download_group([name for name, _ in jobs])
            if win is None:
                win = build_window(os.path.join(RAW_DIR, jobs[0][0]), roi)
                if not os.path.exists(local):
                    create_output(local, "drought", win)
                ds = raw(nc4.Dataset(local, "a"))
                check_window(ds, win)
            results = extract_group(client, jobs, win)

            for name, prefix in jobs:
                r = results[name]
                days = np.asarray(r["days"], "f8")
                with timed("write"):
                    T = np.asarray(ds["time"][:], "f8")
                    new = days[~np.isin(days, T)]
                    if new.size:
                        if T.size and new.min() <= T.max():
                            raise DataIntegrityError(f"{name}: its time axis has days inside the stored record that "
                                                     f"other index files do not have.")
                        ds["time"][T.size:T.size + new.size] = new
                        for qn in [k for k in ds.variables if k.endswith("_qc")]:
                            ds[qn][T.size:T.size + new.size] = np.full(new.size, QC_NODATA, "u1")
                        ds["status"][T.size:T.size + new.size] = np.zeros(new.size, "i1")
                        T = np.concatenate([T, new])
                    rows = np.searchsorted(T, days)
                    mapping = output_names(prefix, [b["name"] for b in r["blocks"]])
                    written = []
                    for b in r["blocks"]:
                        out_name, role = mapping[b["name"]]
                        v = ensure_var(ds, out_name, role, prefix, b, name)
                        if role == "data":
                            v.source_complete = np.int8(0)        # until the whole variable is written and checked
                            q = np.full(T.size, QC_NODATA, "u1")
                            q[rows] = qc_flags(b)
                            ds[out_name + "_qc"][0:T.size] = q
                        if rows.size < T.size:                    # days this file lacks: make them fill again
                            lacking = np.setdiff1d(np.arange(T.size), rows)
                            fill = v.getncattr("_FillValue")
                            for _, row, length in _runs(lacking):
                                v[row:row + length] = np.full((length,) + v.shape[1:], fill, v.dtype)
                        write_block(v, rows, b["path"])
                        written.append((v, rows, b["path"]))
                    ds.sync()
                with timed("read-back check"):
                    for v, rows_, path in written:
                        verify_block(v, rows_, path)
                st = slice_status(r["global"], days.size, default=STATUS_NOT_STATED)
                for d, s in zip(days.tolist(), st.tolist()):
                    status_run[d] = max(status_run.get(d, 0), s)
                info = infos[name]
                for b in r["blocks"]:
                    out_name, role = mapping[b["name"]]
                    ds[out_name].setncatts(dict(source_file=name, source_etag=info["etag"],
                                                source_last_modified=info["last_modified"],
                                                source_size=np.int64(info["size"]),
                                                source_file_date=str(r["global"].get("date", ""))))
                    if role == "data":
                        ds[out_name].source_complete = np.int8(1)
                # status: this run's files decide for the days they cover
                cur = np.asarray(ds["status"][:], "i1")
                fresh = np.array([status_run.get(d, -1) for d in T.tolist()], "i1")
                all_now = all(p in todo for p in prefixes)
                ds["status"][:] = np.where(fresh >= 0, fresh if all_now else np.maximum(cur, fresh), cur)
                ds.committed_records = np.int32(T.size)
                ds.requested_variables = ",".join(sorted(k for k in ds.variables if k + "_qc" in ds.variables))
                ds.date_modified = iso(utcnow())
                ds.time_coverage_start = day_to_date(T[0]).isoformat()
                ds.time_coverage_end = day_to_date(T[-1]).isoformat()
                ds.sync()
                done += 1
                TOTALS["drought_files"] = TOTALS.get("drought_files", 0) + 1
                log.info("Drought %s: %d steps %s -> %s written (%d of %d files) | %.1f GB free", prefix, days.size,
                         day_to_date(days[0]), day_to_date(days[-1]), done, len(todo), free_gb())
            remove_raw([name for name, _ in jobs])
            remove_blocks(results)
            if i >= len(todo):
                ds.processing_status = "complete" if all(_drought_done(ds, p, heads[p]) for p in prefixes) \
                    else "in_progress"
                ds.sync()
            elif time.time() - last_backup >= BACKUP_INTERVAL_S:
                ds.close()
                ds = None
                backup(local, on_drive)
                ds = raw(nc4.Dataset(local, "a"))
                last_backup = time.time()
    except (KeyboardInterrupt, PipelineStop, DataIntegrityError) as e:
        log.warning("Drought stopped (%s). Saving the %d finished index file(s) to Drive.", type(e).__name__, done)
        raise
    finally:
        if ds is not None:
            ds.close()
        if os.path.exists(local) and done:
            backup(local, on_drive)
    return dict(path=on_drive, records=_time_size(local), new_files=done, total_bytes=total_bytes,
                todo_bytes=todo_bytes, seconds=time.time() - t_run, all_files=len(_select(single)))


def run_all():
    """Run both exports. Stops cleanly on Ctrl-C / interrupt: whatever is committed is copied to Drive."""
    TIMES.clear()
    TOTALS.clear()
    t0 = time.time()
    roi = load_roi()
    yearly, single = list_source()
    log.info("Lab folder: %d daily variables %s | %d whole-record files %s", len(yearly), sorted(yearly),
             len(single), sorted(single))
    if ONLY_VARS is not None:
        unknown = [v for v in ONLY_VARS if v not in yearly and v not in single]
        if unknown:
            raise PipelineStop(f"ONLY_VARS names {unknown}, which the lab's folder does not contain.")
    ensure_disk_free("start")
    cluster, client = start_cluster()
    out = {}
    try:
        if RUN_GRIDMET:
            out["gridmet"] = run_gridmet(client, roi, yearly)
        if RUN_DROUGHT:
            out["drought"] = run_drought(client, roi, single)
    except KeyboardInterrupt:
        log.warning("Interrupted. Committed data are on Drive; Run all again to continue.")
    except PipelineStop as e:
        log.error("STOPPED: %s", e)
    finally:
        client.close()
        cluster.close()
        shutil.rmtree(DASK_TMP, ignore_errors=True)
    el = time.time() - t0
    log.info("Run time %.1f min | by stage (s): %s", el / 60, {k: round(v) for k, v in TIMES.items()})
    g, d = out.get("gridmet"), out.get("drought")
    if TEST_MODE:
        for label, r, nkey, allkey in (("gridMET", g, "years", "all_years"), ("Drought", d, "new_files", "all_files")):
            if r and r.get("seconds") and r.get(nkey):
                size = os.path.getsize(r["path"])
                scale = r[allkey] / r[nkey]
                log.info("%s rehearsal: %s downloaded, output %s. Projected full run: about %.0f min at this "
                         "speed, output about %s.", label, human(r["todo_bytes"]), human(size),
                         r["seconds"] * scale / 60, human(size * scale))
        log.info("Rehearsal finished. Set TEST_MODE = False in Cell 2 and Run all for the whole record.")
    return out


RESULT = run_all()

In [ ]:
# ═══════════════ CELL 10: audit of the files on Drive + independent cross-check ═══════════════
def audit_output(path):
    """Reopen the Drive copy and check structure, time axis and bookkeeping. Returns (info, problems)."""
    problems, info = [], {}
    with nc4.Dataset(path) as ds:
        raw(ds)
        n = ds.dimensions["time"].size
        committed = int(getattr(ds, "committed_records", -1))
        if committed != n:
            problems.append(f"{n} records but committed_records = {committed}")
        t = np.asarray(ds["time"][:], "f8")
        if n and np.any(np.diff(t) <= 0):
            problems.append("time is not strictly increasing")
        steps = {float(k): int(c) for k, c in zip(*np.unique(np.diff(t), return_counts=True))} if n > 1 else {}
        st = np.asarray(ds["status"][:])
        data = [k for k in ds.variables if k + "_qc" in ds.variables]
        cats = [k for k in ds.variables if k.endswith("_category")]
        qc = {}
        for k in data:
            v = ds[k]
            for a in ("_FillValue", "units", "long_name", "grid_mapping"):
                if a not in v.ncattrs():
                    problems.append(f"{k}: attribute {a} is missing")
            q = np.asarray(ds[k + "_qc"][:])
            if q.size != n or np.any(q > 7):
                problems.append(f"{k}_qc: unwritten or invalid flags")
            qc[k] = {name: int((q & bit > 0).sum()) for name, bit in
                     (("constant", QC_CONSTANT), ("repeat", QC_REPEAT), ("no_data", QC_NODATA))}
            if str(getattr(ds, "product_kind", "")) == "drought" and int(getattr(v, "source_complete", 0)) != 1:
                problems.append(f"{k}: not completely written (source_complete != 1)")
        info.update(records=n, first=str(day_to_date(t[0])) if n else None, last=str(day_to_date(t[-1])) if n else None,
                    steps=steps, variables=data, categories=len(cats), size=os.path.getsize(path),
                    status={name: int((st == c).sum()) for name, c in
                            (("permanent", 0), ("provisional", 1), ("early", 2), ("not_stated", 3))},
                    processing_status=str(getattr(ds, "processing_status", "?")), qc=qc)
    return info, problems


def _opendap_folder_candidates(prefix):
    stem = re.sub(r"\d.*$", "", prefix)
    return [prefix, stem] if stem and stem != prefix else [prefix]


def cross_check(path, k):
    """Read k random permanent records again through a different route (the lab's OPeNDAP server, which
    subsets on the server) and compare them with the output bit for bit. The server lacks a few variables;
    those are skipped. Returns (checked, skipped)."""
    checked = skipped = 0
    rng = random.Random(12345)
    with nc4.Dataset(path) as ds:
        raw(ds)
        kind = str(getattr(ds, "product_kind", ""))
        t = np.asarray(ds["time"][:], "f8")
        st = np.asarray(ds["status"][:])
        r0, r1 = [int(x) for x in str(ds.window_rows).split(":")]
        c0, c1 = [int(x) for x in str(ds.window_cols).split(":")]
        data = [v for v in ds.variables if v + "_qc" in ds.variables]
        picks = []
        for _ in range(k * 6):
            name = rng.choice(data)
            ok = np.where(((st == STATUS_PERMANENT) | (st == STATUS_NOT_STATED))
                          & (np.asarray(ds[name + "_qc"][:]) & QC_NODATA == 0))[0]
            if ok.size:
                picks.append((name, int(rng.choice(ok.tolist()))))
        for name, i in picks:
            if checked >= k:
                break
            src_var = str(ds[name].source_variable)
            fname = f"{name}_{year_of(t[i])}.nc" if kind == "gridmet" else f"{name}.nc"
            remote = None
            for folder in _opendap_folder_candidates(name):
                try:
                    remote = nc4.Dataset(f"{OPENDAP_URL}{folder}/{fname}")
                    break
                except Exception:
                    remote = None
            if remote is None:
                skipped += 1
                continue
            try:
                remote.set_auto_maskandscale(False)
                j = np.where(np.asarray(remote["day"][:], "f8") == t[i])[0]
                if j.size != 1:
                    skipped += 1
                    continue
                a = np.asarray(remote[src_var][int(j[0]), r0:r1, c0:c1])
            except Exception as e:
                log.warning("cross-check: %s unreadable through OPeNDAP (%s)", fname, str(e)[:80])
                skipped += 1
                continue
            finally:
                remote.close()
            b = np.asarray(ds[name][i])
            if a.dtype != b.dtype and a.dtype.itemsize == b.dtype.itemsize:
                a = a.view(b.dtype)                        # OPeNDAP shows unsigned 16-bit as signed
            if not np.array_equal(a, b):
                raise DataIntegrityError(f"Cross-check FAILED: {name} on {day_to_date(t[i])} differs from the lab's "
                                         f"OPeNDAP copy in {int((a != b).sum())} cells.")
            checked += 1
            log.info("cross-check OK: %-9s %s identical to the lab's OPeNDAP copy", name, day_to_date(t[i]))
    return checked, skipped


for _name in (GRIDMET_NAME, DROUGHT_NAME):
    _p = os.path.join(DRIVE_OUT, _name)
    if not os.path.exists(_p):
        continue
    _info, _problems = audit_output(_p)
    print(f"\n{_name}: {human(_info['size'])}, {_info['records']} records {_info['first']} -> {_info['last']}, "
          f"{len(_info['variables'])} variables + {_info['categories']} category layers, "
          f"status {_info['status']}, {_info['processing_status']}")
    print("  time steps (days: count):", _info["steps"])
    _odd = {v: q for v, q in _info["qc"].items() if any(q.values())}
    print("  records flagged at the source:", _odd if _odd else "none")
    for _x in _problems:
        print("  PROBLEM:", _x)
    try:
        _c, _s = cross_check(_p, CROSS_CHECKS)
        print(f"  cross-check through OPeNDAP: {_c} records identical, {_s} skipped (not served there)")
    except DataIntegrityError:
        raise
    except Exception as _e:
        print("  cross-check not possible:", str(_e)[:120])
    if _info["processing_status"] != "complete":
        print("  NOT FINISHED: the last run stopped early. Run all again to continue.")
    print("  AUDIT", "PASSED" if not _problems else "FAILED")